# Lab: Patterns in ERA5

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/evanwellmeyer/teaching/blob/main/ai_weather_lab/unsupervised_era5.ipynb)

Unsupervised machine learning finds structure in data without being told what to look for. Here you take 64 years of ERA5 and break it into simple pieces: a time series into waves (Fourier), a field into the patterns the data itself picks out (EOFs), and a field into fixed waves on the sphere (spherical harmonics).

A CPU runtime is enough. Run the two setup cells first.

In [ ]:
#@title Setup: install packages (~1 min)
!pip install -q gcsfs cartopy

In [ ]:
#@title Setup: imports and ERA5 data
import time

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.special
import xarray as xr
from IPython.display import display

plt.rcParams.update({"font.size": 11, "figure.dpi": 100})
PC = ccrs.PlateCarree()   # longitude-latitude coordinates; pass transform=PC when you draw data on a map
G = 9.80665               # gravity, m/s^2

# ERA5 every 6 hours from 1959 to 2022 on a coarse grid (64 x 32 points, 5.6 degrees apart).
# This is WeatherBench 2's copy on Google Cloud; no account needed.
ERA5 = xr.open_zarr("gs://weatherbench2/datasets/era5/1959-2023_01_10-6h-64x32_equiangular_conservative.zarr",
                    storage_options=dict(token="anon")).sel(time=slice("1959", "2022"))


def pretty_units(units):
    return units.replace("**-1", "⁻¹").replace("**-2", "⁻²").replace("**2", "²")


def show_era5_variables(search=""):
    # A table of the ERA5 variables (that change in time) whose name or description contains `search`
    rows = []
    for name in ERA5.data_vars:
        if "time" not in ERA5[name].dims:
            continue
        description = ERA5[name].attrs.get("long_name", "")
        if search.lower() in (name + " " + description).lower():
            rows.append({
                "variable": name,
                "description": description,
                "units": pretty_units(ERA5[name].attrs.get("units", "")),
                "pressure levels": "yes (slow: a few minutes to load)" if "level" in ERA5[name].dims else "",
            })
    table = pd.DataFrame(rows).sort_values(["pressure levels", "variable"])
    try:   # in Colab: a table you can sort and search
        from google.colab import data_table
        display(data_table.DataTable(table, include_index=False, num_rows_per_page=25))
    except ImportError:
        with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
            display(table.set_index("variable"))


def in_plot_units(field, variable, level):
    # Convert ERA5 units to friendlier ones, and keep a name and units for labels
    units = field.attrs.get("units", "")
    name = field.attrs.get("long_name", variable.replace("_", " "))
    if variable.startswith("total_precipitation"):   # m of water over 6, 12 or 24 hours -> mm/day
        hours = int(variable.split("_")[-1].replace("hr", ""))
        field, units = field * 1000 * 24 / hours, "mm/day"
    elif units == "K":
        field, units = field - 273.15, "°C"
    elif units == "Pa":
        field, units = field / 100, "hPa"
    elif units == "m**2 s**-2":
        field, units, name = field / G, "m", name.replace("Geopotential", "Geopotential height")
    elif units == "kg kg**-1":
        field, units = field * 1000, "g/kg"
    if level is not None:
        name = f"{level} hPa {name}"
    return field.assign_attrs(long_name=name, units=pretty_units(units))


MONTHLY = {}   # monthly means already loaded, so they load only once


def load_monthly(variable, level=None):
    # Monthly means of one ERA5 variable, 1959-2022, at pressure `level` (hPa) if it has levels
    if (variable, level) in MONTHLY:
        return MONTHLY[(variable, level)]
    if variable not in ERA5 or "time" not in ERA5[variable].dims:
        raise ValueError(f"ERA5 has no variable {variable!r} that changes in time. "
                         "Run show_era5_variables() to see the names.")
    field = ERA5[variable]
    if "level" in field.dims:
        if level is None:
            raise ValueError(f"{variable} is on pressure levels: give it a level, one of {list(ERA5.level.values)}")
        print(f"Loading {variable} at {level} hPa (variables on pressure levels take a few minutes)...")
        field = field.sel(level=level, drop=True)
    else:
        print(f"Loading {variable}...")
    tic = time.time()
    monthly = field.resample(time="MS").mean().compute()   # average the 6-hourly values in each month
    monthly = in_plot_units(monthly.transpose("time", "latitude", "longitude"), variable, level)
    print(f"Loaded {monthly.sizes['time']} months in {time.time() - tic:.0f} s")
    MONTHLY[(variable, level)] = monthly
    return monthly


def crop(ds, lats, lons):
    # Cut out a box: lats = (south, north), lons = (west, east), with negative longitudes to the west.
    # Longitudes in the result run continuously from west to east, even across 0 or 180 degrees.
    west, east = lons
    width = (east - west) % 360 or 360   # a width of 0 means the whole way around
    offset = (ds.longitude - west) % 360
    ds = ds.isel(longitude=np.flatnonzero(offset.values <= width))
    ds = ds.assign_coords(longitude=west + (ds.longitude - west) % 360).sortby("longitude")
    inside = (ds.latitude.values >= lats[0]) & (ds.latitude.values <= lats[1])
    return ds.isel(latitude=np.flatnonzero(inside)).sortby("latitude")


def box_projection(lats, lons):
    # Map projection centered on a box, and the box's extent in it
    width = (lons[1] - lons[0]) % 360 or 360
    center = lons[0] + width / 2
    return ccrs.PlateCarree(central_longitude=center), [-width / 2, width / 2, lats[0], lats[1]]


def area_mean(field, lats, lons):
    # Average over a box, weighting each grid point by its area (cos of latitude)
    box = crop(field, lats, lons)
    return box.weighted(np.cos(np.deg2rad(box.latitude))).mean(("latitude", "longitude")).assign_attrs(field.attrs)


def anomalies(field):
    # Each value minus the 1959-2022 average for its calendar month: removes the seasonal cycle
    climatology = field.groupby("time.month").mean()
    return (field.groupby("time.month") - climatology).drop_vars("month").assign_attrs(field.attrs)


def remove_trend(field):
    # Subtract a straight-line fit in time at every grid point
    years = (field.time - field.time[0]) / np.timedelta64(365, "D")
    slope = ((years - years.mean()) * (field - field.mean("time"))).mean("time") / years.var()
    return (field - slope * (years - years.mean())).assign_attrs(field.attrs)


def to_matrix(anomaly):
    # The data as a matrix: one row per month, one column per grid point. Each point is weighted by
    # sqrt(cos(latitude)) so that it counts in proportion to its area. Points with missing values
    # (such as sea surface temperature over land) are left out.
    stacked = anomaly.stack(point=("latitude", "longitude")).dropna("point", how="any")
    return (stacked * np.sqrt(np.cos(np.deg2rad(stacked.latitude)))).values


try:
    from scipy.special import sph_harm_y
except ImportError:   # older SciPy
    def sph_harm_y(l, m, polar, azimuth):
        return scipy.special.sph_harm(m, l, azimuth, polar)


def spherical_harmonics(lmax, latitude, longitude):
    # Real spherical harmonics up to total wavenumber lmax on a lat-lon grid: a matrix with one row per grid point
    # (latitude by latitude) and one column per mode, plus the total (l) and zonal (m) wavenumber of each mode.
    # m > 0 goes like cos(m * longitude) and m < 0 like sin(|m| * longitude).
    lat, lon = np.meshgrid(np.deg2rad(latitude), np.deg2rad(longitude), indexing="ij")
    polar, azimuth = np.pi / 2 - lat.ravel(), lon.ravel()
    columns, ls, ms = [], [], []
    for l in range(lmax + 1):
        for m in range(-l, l + 1):
            y = sph_harm_y(l, abs(m), polar, azimuth)
            if m == 0:
                columns.append(y.real)
            elif m > 0:
                columns.append(np.sqrt(2) * y.real)
            else:
                columns.append(np.sqrt(2) * y.imag)
            ls.append(l)
            ms.append(m)
    return np.stack(columns, axis=1), np.array(ls), np.array(ms)


print(f"ERA5: {ERA5.sizes['time']} times from {str(ERA5.time.values[0])[:10]} to {str(ERA5.time.values[-1])[:10]}, "
      f"{ERA5.sizes['longitude']} x {ERA5.sizes['latitude']} grid points")

---
## 1. A time series and its frequencies

Sea surface temperature in the central equatorial Pacific, the Niño 3.4 region (5°S–5°N, 170°W–120°W), is the standard measure of El Niño and La Niña. Here it is from ERA5, averaged over each month.

In [ ]:
sst = load_monthly("sea_surface_temperature")
nino34 = area_mean(sst, lats=(-5, 5), lons=(-170, -120))

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(nino34.time, nino34, color="black", linewidth=0.8)
ax.set_ylabel("°C")
ax.set_title("Sea surface temperature in the Niño 3.4 region")
plt.show()

A Fourier transform rewrites a time series as a sum of sine waves, one for each frequency, and gives how much of the series each wave carries (its power). The period of a wave is one over its frequency.

In [ ]:
values = nino34.values - nino34.values.mean()
power = np.abs(np.fft.rfft(values)) ** 2            # how strong each wave is
frequency = np.fft.rfftfreq(len(values), d=1 / 12)  # cycles per year (one value per month)
period = 1 / frequency[1:]                          # years per cycle (frequency 0 is just the mean)

ticks = [0.25, 0.5, 1, 2, 4, 8, 16, 32]
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(period, power[1:], color="black")
ax.set_xscale("log")
ax.set_xticks(ticks, [f"{t:g}" for t in ticks])
ax.set_xlabel("period (years)")
ax.set_ylabel("power")
ax.set_title("Fourier spectrum of Niño 3.4 sea surface temperature")
plt.show()

The tallest peak is the seasonal cycle, at one cycle per year. Remove it by subtracting the average for each calendar month (the anomaly), and take the spectrum again.

In [ ]:
nino34_anomaly = anomalies(nino34)
values = nino34_anomaly.values
power = np.abs(np.fft.rfft(values)) ** 2

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), layout="constrained")
ax1.fill_between(nino34_anomaly.time, values, where=values > 0, color="tab:red")
ax1.fill_between(nino34_anomaly.time, values, where=values < 0, color="tab:blue")
ax1.set_ylabel("°C")
ax1.set_title("Niño 3.4 anomaly: El Niño (red) and La Niña (blue)")
ax2.plot(period, power[1:], color="black")
ax2.axvspan(2, 7, color="0.9")   # the usual range of El Niño periods
ax2.set_xscale("log")
ax2.set_xticks(ticks, [f"{t:g}" for t in ticks])
ax2.set_xlabel("period (years)")
ax2.set_ylabel("power")
ax2.set_title("Fourier spectrum of the anomaly (gray: 2 to 7 years)")
plt.show()

Rebuild the anomaly from only its strongest waves. How many do you need?

In [ ]:
N_WAVES = 10

coefficients = np.fft.rfft(values)
strongest = np.argsort(np.abs(coefficients))[::-1][:N_WAVES]   # the N_WAVES largest
kept = np.zeros_like(coefficients)
kept[strongest] = coefficients[strongest]
rebuilt = np.fft.irfft(kept, n=len(values))
explained = 1 - np.var(values - rebuilt) / np.var(values)

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(nino34_anomaly.time, values, color="0.6", linewidth=0.8, label="anomaly")
ax.plot(nino34_anomaly.time, rebuilt, color="black", linewidth=1.5, label=f"{N_WAVES} strongest waves")
ax.set_ylabel("°C")
ax.set_title(f"{N_WAVES} of {len(coefficients)} waves explain {explained:.0%} of the variance")
ax.legend(loc="upper left")
plt.show()

---
## 2. Patterns the data finds: EOFs

Sine waves are chosen in advance. Empirical orthogonal functions (EOFs, also called principal component analysis) let the data choose. The first EOF is the map pattern that explains the most variance over all the months, the second explains the most of what is left, and so on. Each comes with a time series, its principal component (PC), that says how strongly the pattern is present in each month. The method is never told what El Niño is.

Pick one or two variables from the table, a box, and the months to use. With two variables, both count equally and each mode has a pattern in each.

In [ ]:
show_era5_variables()

In [ ]:
VARIABLES = [("sea_surface_temperature", None)]   # (variable, pressure level or None); one or two of them
LATS = (-30, 30)       # south, north
LONS = (120, -70)      # west, east (negative = west of Greenwich); (0, 0) is the whole way around
MONTHS = "all"         # or a list, e.g. [12, 1, 2] for northern winter
N_MODES = 3
REMOVE_TREND = True    # remove the straight-line trend at every point (otherwise warming can be a mode)

# Other things to try:
# North Atlantic Oscillation:   VARIABLES = [("mean_sea_level_pressure", None)], LATS = (20, 80), LONS = (-90, 40), MONTHS = [12, 1, 2]
# El Niño in sea and air:       VARIABLES = [("sea_surface_temperature", None), ("mean_sea_level_pressure", None)], LATS = (-30, 30), LONS = (90, -60)
# Global warming:               VARIABLES = [("2m_temperature", None)], LATS = (-90, 90), LONS = (0, 0), REMOVE_TREND = False
# Northern annular mode (slow): VARIABLES = [("geopotential", 500)], LATS = (20, 90), LONS = (0, 0), MONTHS = [12, 1, 2]

anomaly = {}
for variable, level in VARIABLES:
    field = anomalies(crop(load_monthly(variable, level), LATS, LONS))
    if MONTHS != "all":
        field = field.sel(time=field.time.dt.month.isin(MONTHS))
    if REMOVE_TREND:
        field = remove_trend(field)
    anomaly[field.attrs["long_name"]] = field
times = field.time

# The data matrix: one row per month, one column per grid point (of each variable).
# Each variable is divided by its overall standard deviation, so that two variables count equally.
X = np.concatenate([to_matrix(field) / float(field.std()) for field in anomaly.values()], axis=1)
print(f"{X.shape[0]} months x {X.shape[1]} grid points")

# The EOF analysis itself: a singular value decomposition, X = U S V^T
U, S, VT = np.linalg.svd(X, full_matrices=False)
explained = S ** 2 / np.sum(S ** 2)   # the fraction of the variance each mode explains
pcs = U * S                           # principal components: one time series per mode
pcs = pcs / pcs.std(axis=0)           # in units of standard deviations

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(np.arange(1, 21), 100 * explained[:20], color="0.4")
ax.set_xticks(np.arange(1, 21))
ax.set_xlabel("mode")
ax.set_ylabel("% of variance")
ax.set_title("Variance explained by each mode")
plt.show()

Each map shows the anomaly that goes with one standard deviation of that mode's PC (a regression map), in the variable's units. The sign of a mode is arbitrary: the pattern times the PC is what matters.

In [ ]:
proj, extent = box_projection(LATS, LONS)
columns = len(anomaly) + 1
fig = plt.figure(figsize=(6 * columns, 3.2 * N_MODES), layout="constrained")
for k in range(N_MODES):
    pc = xr.DataArray(pcs[:, k], coords={"time": times})
    for j, (name, field) in enumerate(anomaly.items()):
        pattern = (field * pc).mean("time")   # regression of the anomaly on the PC
        top = float(np.abs(pattern).max())
        ax = fig.add_subplot(N_MODES, columns, k * columns + j + 1, projection=proj)
        mesh = ax.pcolormesh(pattern.longitude, pattern.latitude, pattern, transform=PC,
                             cmap="RdBu_r", vmin=-top, vmax=top)
        ax.coastlines(linewidth=0.6)
        ax.set_extent(extent, crs=proj)
        fig.colorbar(mesh, ax=ax, shrink=0.8, label=field.attrs["units"])
        ax.set_title(f"EOF {k + 1} ({explained[k]:.0%}): {name}", fontsize=10)
    ax = fig.add_subplot(N_MODES, columns, k * columns + columns)
    ax.plot(times, pc, color="black", linewidth=0.7)
    ax.axhline(0, color="0.6", linewidth=0.5)
    ax.set_title(f"PC {k + 1}", fontsize=10)
plt.show()

Compare PC 1 with the Niño 3.4 anomaly from Part 1.

In [ ]:
pc1 = xr.DataArray(pcs[:, 0], coords={"time": times})
nino = nino34_anomaly.sel(time=times)
nino = (nino - nino.mean()) / nino.std()
r = np.corrcoef(pc1, nino)[0, 1]

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(times, nino, color="tab:red", linewidth=1, label="Niño 3.4 anomaly")
ax.plot(times, np.sign(r) * pc1, color="black", linewidth=1, label="PC 1" + (" (times -1)" if r < 0 else ""))
ax.set_ylabel("standard deviations")
ax.set_title(f"Correlation: {r:.2f}")
ax.legend(loc="upper left")
plt.show()

---
## 3. Patterns chosen in advance: spherical harmonics

On a sphere, the natural waves are spherical harmonics, the two-dimensional relatives of sines and cosines. Each has a total wavenumber l (larger l, smaller features) and a zonal wavenumber m (the number of waves around a latitude circle). Like Fourier waves, they are the same for every field. Here are a few.

In [ ]:
LMAX = 21   # the largest total wavenumber to use
lat, lon = ERA5.latitude.values, ERA5.longitude.values
Y, l_of, m_of = spherical_harmonics(LMAX, lat, lon)
print(f"{Y.shape[1]} modes on {Y.shape[0]} grid points")

SHOW = [(0, 0), (1, 0), (1, 1), (2, 0), (3, 2), (6, 6), (10, 3), (20, 0)]   # (l, m)
fig = plt.figure(figsize=(16, 6), layout="constrained")
for i, (l, m) in enumerate(SHOW):
    mode = Y[:, (l_of == l) & (m_of == m)].reshape(len(lat), len(lon))
    top = np.abs(mode).max()
    ax = fig.add_subplot(2, 4, i + 1, projection=ccrs.Robinson(central_longitude=180))
    ax.pcolormesh(lon, lat, mode, transform=PC, cmap="RdBu_r", vmin=-top, vmax=top)
    ax.coastlines(linewidth=0.4, color="0.3")
    ax.set_title(f"l = {l}, m = {m}")
plt.show()

Any global field is a sum of these modes. Pick a field and find how much of each mode it contains, with a least-squares fit. The power at each l shows how the field's variance is spread from large scales (small l) to small ones. The field must have values everywhere, so sea surface temperature does not work here.

In [ ]:
VARIABLE = "2m_temperature"
LEVEL = None          # only for variables on pressure levels
WHEN = "mean"         # "mean" for the 1959-2022 average, or one month, e.g. "2015-12"
ANOMALY = False       # True: use the anomaly (the field minus the average for its calendar month)

field = load_monthly(VARIABLE, LEVEL)
if ANOMALY:
    field = anomalies(field)
field = field.mean("time").assign_attrs(field.attrs) if WHEN == "mean" else field.sel(time=WHEN).squeeze()

# Least squares, with each grid point weighted by its area (cos of latitude)
values = field.values.ravel()
weights = np.sqrt(np.cos(np.deg2rad(lat))).repeat(len(lon))
coefficients = np.linalg.lstsq(Y * weights[:, None], values * weights, rcond=None)[0]
power = np.array([np.sum(coefficients[l_of == l] ** 2) for l in range(LMAX + 1)])

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(np.arange(1, LMAX + 1), power[1:], "o-", color="black")   # l = 0 is the global mean
ax.set_xticks(range(0, LMAX + 1, 2))
ax.set_xlabel("total wavenumber l")
ax.set_ylabel("power")
ax.set_title(f"Spherical harmonic spectrum: {field.attrs['long_name']}")
plt.show()

Now rebuild the field from more and more modes, starting from the largest scales.

In [ ]:
TRUNCATIONS = [1, 2, 4, 8, LMAX]   # keep the modes with l up to each of these

area = np.cos(np.deg2rad(lat))[:, None] * np.ones(len(lon))
lo, hi = np.percentile(values, [1, 99])
cmap = "viridis"
if lo < 0 < hi:   # values of both signs (e.g. an anomaly): blue below zero, red above
    cmap, hi = "RdBu_r", max(-lo, hi)
    lo = -hi
fig = plt.figure(figsize=(16, 6.5), layout="constrained")
ax = fig.add_subplot(2, 3, 1, projection=ccrs.Robinson(central_longitude=180))
mesh = ax.pcolormesh(lon, lat, field, transform=PC, cmap=cmap, vmin=lo, vmax=hi)
ax.coastlines(linewidth=0.4)
ax.set_title(f"ERA5: {field.attrs['long_name']}", fontsize=10)
for i, L in enumerate(TRUNCATIONS):
    keep = l_of <= L
    rebuilt = (Y[:, keep] @ coefficients[keep]).reshape(len(lat), len(lon))
    error = field.values - rebuilt
    explained = 1 - np.average(error ** 2, weights=area) / np.average((field.values - np.average(field.values, weights=area)) ** 2, weights=area)
    ax = fig.add_subplot(2, 3, i + 2, projection=ccrs.Robinson(central_longitude=180))
    ax.pcolormesh(lon, lat, rebuilt, transform=PC, cmap=cmap, vmin=lo, vmax=hi)
    ax.coastlines(linewidth=0.4)
    ax.set_title(f"l up to {L}: {keep.sum()} modes, {max(explained, 0):.0%} of the variance", fontsize=10)
fig.colorbar(mesh, ax=fig.axes, orientation="horizontal", shrink=0.5, label=field.attrs["units"])
plt.show()